In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

In [6]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [7]:
question = 'How does the agentic loop keep calling the model until it stops?'
search_result = index.search(
    question, 
    num_results=5
)

search_result[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [15]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question} 

CONTEXT:
{context}
""".strip()

In [21]:
class RAGBase:
    def __init__(
            self,
            index,
            llm_client,
            instructions = INSTRUCTIONS,
            prompt_template = PROMPT_TEMPLATE,
            model = 'gpt-5.4-mini'
        ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model 
    
    def search(self, question):
        return self.index.search(
            question, 
            num_results = 5
        )
    
    def build_context(self, search_result):
        lines = []
        for doc in search_result:
            lines.append('filename:' + doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()
    
    def build_prompt(self, question, search_results):
        context = self.build_context(search_results)
        prompt = self.prompt_template.format(
            question=question,
            context=context
        )
        return prompt.strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage
    
    def rag(self,query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, usage = self.llm(prompt)
        return answer, usage

In [22]:
assistant = RAGBase(
    index = index,
    llm_client=openai_client
)

query = 'How does the agentic loop keep calling the model until it stops?'
answer, usage = assistant.rag(query)


In [23]:
usage.input_tokens

7131

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
len(chunks)

295

In [27]:
index_chunk = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index_chunk.fit(chunks)

In [28]:
assistant = RAGBase(
    index = index_chunk,
    llm_client=openai_client
)

query = 'How does the agentic loop keep calling the model until it stops?'
answer, usage = assistant.rag(query)


In [29]:
usage.input_tokens

2314

In [30]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()



In [32]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [34]:
def search(query: str) -> dict[str, str]:
    """
    Search the lesson content for entries matching the given query.
    """
    return index_chunk.search(
        query,
        num_results=5,
    )

In [35]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [37]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
